# LLM Fine-Tuning Deep Dive, Part 3 of 3: Comparison & Decision

> **Learning objective:** learn how to compare fine-tuned models without letting one impressive output make the decision for you.

Parts 1 and 2 produced several candidates. Part 3 asks: **which differences matter for Riverside's actual job?**

> **Mechanics revision:** fixed-token likelihood and checkpoint comparisons rest on the same [per-position loss](../02-transformers/02-decoder-only-language-model.ipynb#per-position-loss-one-sequence-many-lessons) and [one-update](../02-transformers/02-decoder-only-language-model.ipynb#one-backward-pass-many-token-lessons-one-update) mechanics established upstream. For seq2seq contrast, revisit the [teacher-forced target trace](../02-transformers/03-encoder-decoder-and-cross-attention.ipynb#teacher-forced-inner-mechanics-source-once-target-positions-together). Part 3 measures evidence; it does not re-derive training.

## Start with the Simple Problem

One continuation can look excellent because generation varies from run to run. A fairer comparison does three things:

1. give the models the same text to score;
2. repeat the check across more Riverside passages;
3. decide what result would be good enough for the workload.

![A Riverside manuscript card moving through four visual stations: one variable generated continuation, fixed-token probability comparison, a stack of held-out passages, and a workload decision gate](images/evaluation-evidence-ladder.png)

## Hardware Profile Boundary

Parts 1–3 use one shared profile and one profile-specific checkpoint root:

- CPU: SmolLM2-135M / 135M-Instruct.
- CUDA below 64 GiB: SmolLM2-360M / 360M-Instruct.
- CUDA with at least 64 GiB: SmolLM2-1.7B / 1.7B-Instruct.

The CPU path is expected to produce weaker and sometimes indistinguishable text. A 135M model, a narrow corpus, stochastic decoding, and ten-step teaching runs can leave visible generations unchanged even when weights or probabilities move. That is why this notebook reports objective-aligned metrics and allows an `INCONCLUSIVE` decision instead of presenting small-model output as production evidence.

## Keep the Two Fine-Tuning Choices Separate

```mermaid
flowchart LR
    subgraph Behavior["What should the model learn?"]
        direction TB
        B0["Base model<br/>general language"]
        B1["Continued pretraining<br/>domain prose"]
        B2["SFT<br/>instruction contract"]
        B3["DPO<br/>editor preference"]
        B0 --> B1 --> B2 --> B3
    end

    subgraph Parameters["Where should the update live?"]
        direction TB
        P0["Full fine-tuning<br/>all weights"]
        P1["Partial freezing<br/>selected layers"]
        P2["LoRA<br/>small adapters"]
        P3["QLoRA<br/>compact frozen base + adapters"]
        P0 --> P1 --> P2 --> P3
    end
```

SFT and LoRA answer different questions: SFT defines the behavior being taught; LoRA defines how its update is stored. The notebook follows one intuitive loop:

> What went wrong? -> What would reveal improvement? -> What result changes the decision?

| Stage | Riverside asks | What to inspect |
| --- | --- | --- |
| Continued pretraining | Does Riverside prose feel less unexpected to the model? | The same continuation first, then many held-out passages |
| SFT | Does the assistant follow the request completely? | Pass/fail checks for every required rule |
| DPO | Do editors prefer it to the SFT answer? | Blinded wins, losses, and ties |
| Parameter strategy | Does the cheaper update preserve the needed behavior? | Workload quality first, measured cost second |
| Release | Is it good and practical enough to operate? | Quality, safety, latency, cost, and rollback readiness |

---

## Setup: Reload the Six Candidates

Parts 1 and 2 saved independent candidates under `checkpoints/llm-finetuning/<profile>/`. This notebook reloads fresh objects so one adapter cannot alter another candidate's base.

> **Prerequisite:** rerun Parts 1 and 2 from clean kernels on the same hardware profile before loading candidates here.

The candidates differ in what they learned, how their updates were stored, and which data they saw. Their raw numbers should therefore answer narrow questions, not form one universal leaderboard.

## Use the Measurement That Matches the Job

Start with likelihood because continued pretraining practiced next-token prediction. Change the measurement when the job changes.

| Training goal | First useful question | What to count |
| --- | --- | --- |
| Continued pretraining | Does the model expect held-out Riverside prose more strongly? | NLL/perplexity on the same text |
| SFT | Does it follow every required instruction rule? | Complete passes across held-out requests |
| DPO | Do editors choose it over SFT? | Blinded wins, losses, and ties |
| Parameter strategy | Does a cheaper update keep the required behavior? | The same behavior check, then actual resource cost |

The walking sentence makes the first progression tangible: look at one generated continuation, score the same continuation under both models, then repeat across many passages. Because these checkpoints were trained differently, the results show what to investigate next rather than one universal winner.

### Candidate Manifest: What Will Be Reloaded?

The notebook needs six independent model objects so that loading one adapter cannot mutate another candidate's base model.

| Candidate | Saved artifact | Objective | Parameter strategy | Ancestry |
| --- | --- | --- | --- | --- |
| Baseline | Hugging Face base checkpoint | Original pretraining | No Riverside update | SmolLM2 base |
| Full-FT continuation | `non-instruction-full` | Continued pretraining | Full fine-tuning | SmolLM2 base |
| Partial-freeze continuation | `partial-freeze` | Continued pretraining | Selected late layers | SmolLM2 base |
| LoRA continuation | `peft-lora` | Continued pretraining | LoRA | Fresh SmolLM2 base + adapter |
| SFT assistant | `instruction-lora` | SFT | LoRA | Fresh SmolLM2 base + adapter |
| DPO assistant | `preference-dpo` | DPO | Continue SFT LoRA | Fresh SmolLM2 base + DPO adapter |

The code first restores the common tokenizer and prompt contract, then loads a fresh base for every PEFT adapter, verifies the LoRA target modules, and derives parameter counts from the objects actually loaded. This reconstructs the experiment before any comparison begins.

> **PyTorch → Keras:** `torch.cuda.is_available()` + `.to(device)` explicitly move a model/tensors to
> GPU or CPU, `AutoModelForCausalLM.from_pretrained(...)` loads pretrained weights, and
> `model.generate(...)` run inside `torch.no_grad()` performs autoregressive decoding without tracking
> gradients (nothing to backprop through during inference). **Keras/TF equivalent:** TensorFlow places
> ops on GPU automatically (explicit placement is `tf.device(...)`, rarely needed); the loading call
> would be `TFAutoModelForCausalLM.from_pretrained(...)` followed by the same `.generate(...)` method --
> Keras/TF has no separate "no_grad" context since inference doesn't build a gradient tape by default.


In [ ]:
# Re-establish Parts 1-2's hardware-aware SmolLM2 profile and chat contract.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA_AVAILABLE else 0.0
)

if not CUDA_AVAILABLE:
    MODEL_PROFILE = "cpu-small-135m"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    INSTRUCT_MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
elif GPU_MEMORY_GIB < 64:
    MODEL_PROFILE = "gpu-balanced-360m"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
    INSTRUCT_MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
else:
    MODEL_PROFILE = "gpu-quality-1.7b"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    INSTRUCT_MODEL_REVISION = "31b70e2e869a7173562077fd711b654946d38674"

MODEL_NAME = INSTRUCT_MODEL_NAME
MODEL_REVISION = INSTRUCT_MODEL_REVISION
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
CONTINUATION_INSTRUCTION = "Continue the fiction narrative in the same style."

try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

REPO_ROOT = _notebook_dir.parents[2]
CHECKPOINT_ROOT = REPO_ROOT / "checkpoints"
CHECKPOINT_DIR = CHECKPOINT_ROOT / "llm-finetuning" / MODEL_PROFILE

device = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"Using device: {device} ({MODEL_PROFILE}, {GPU_MEMORY_GIB:.1f} GiB CUDA memory)")
print(f"Profile checkpoints: {CHECKPOINT_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"

num_hidden_layers = len(base_model.model.layers)
hidden_size = base_model.config.hidden_size
total_base_parameters = sum(parameter.numel() for parameter in base_model.parameters())
print(
    f"Loaded {MODEL_NAME}@{MODEL_REVISION[:8]}: {total_base_parameters:,} parameters, "
    f"{num_hidden_layers} decoder layers, hidden size {hidden_size}."
)
if not CUDA_AVAILABLE:
    print(
        "CPU disclaimer: the 135M candidates may all remain generic or visibly similar after "
        "short teaching runs. Treat that as an expected capacity-and-budget result; compare "
        "reserved metrics, artifact size, and parameter cost without claiming production quality."
    )


def instruction_prompt(prompt):
    """Build the user request used by the SFT and DPO recipes."""
    return f"{CONTINUATION_INSTRUCTION}\n\nContext:\n{prompt}"


def apply_instruction_template(prompt):
    """Serialize a request through SmolLM2's native chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt.strip()},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate(model, prompt, max_new_tokens=60, use_instruction_template=False):
    """Generate only new tokens, formatting instruction candidates consistently."""
    model.eval()
    model_input = apply_instruction_template(prompt) if use_instruction_template else prompt
    model_device = next(model.parameters()).device
    inputs = tokenizer(model_input, return_tensors="pt")
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        output_ids[0][prompt_length:], skip_special_tokens=True
    ).strip()
    return completion if completion else "[model stopped immediately after the prompt]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")

### Reloading the Five Fine-Tuned Checkpoints

Each PEFT-wrapped adapter (instruction-tuned LoRA, DPO, LoRA continued pretraining) gets its own fresh
base-model instance rather than sharing `base_model` above -- the same "every PEFT wrapper gets its
own base" rule Parts 1-2 followed throughout. `freeze_model`'s `requires_grad` flags are re-applied
after loading (see the comment below) since that bookkeeping isn't part of a saved checkpoint -- only
the trained weights are.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, path)` wraps a fresh base model with a
> saved LoRA adapter's weights; `named_parameters()` iterates `(name, tensor)` pairs so `requires_grad`
> can be toggled per-parameter (used here to re-apply the freeze pattern, since that bookkeeping isn't
> part of a saved checkpoint), and `p.numel()` counts a tensor's elements to total trainable params.
> **Keras/TF equivalent:** LoRA loading has no single standard TF API (usually a custom `tf.keras.Model`
> subclass or a TF-specific PEFT integration); freezing is coarser-grained -- `layer.trainable = False`
> per layer rather than per-parameter -- and element counts come from `tf.size(variable)`.


In [ ]:
# Reload only artifacts regenerated by Parts 1-2 for MODEL_NAME.

required_checkpoint_artifacts = {
    CHECKPOINT_DIR / "non-instruction-full": ("config.json", "model.safetensors"),
    CHECKPOINT_DIR / "instruction-lora": ("adapter_config.json", "adapter_model.safetensors"),
    CHECKPOINT_DIR / "preference-dpo": ("adapter_config.json", "adapter_model.safetensors"),
    CHECKPOINT_DIR / "partial-freeze": ("config.json", "model.safetensors"),
    CHECKPOINT_DIR / "peft-lora": ("adapter_config.json", "adapter_model.safetensors"),
}
missing_checkpoint_artifacts = [
    checkpoint_dir / artifact_name
    for checkpoint_dir, artifact_names in required_checkpoint_artifacts.items()
    for artifact_name in artifact_names
    if not (checkpoint_dir / artifact_name).is_file()
]
if missing_checkpoint_artifacts:
    missing_lines = "\n".join(f"  - {path}" for path in missing_checkpoint_artifacts)
    raise FileNotFoundError(
        f"Missing required fine-tuning artifacts:\n{missing_lines}\n"
        "Run Parts 1 and 2 first to generate these checkpoints."
    )

non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR / "non-instruction-full"
).to(device)

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, CHECKPOINT_DIR / "instruction-lora"
).to(device)

dpo_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model = PeftModel.from_pretrained(
    dpo_base_reload, CHECKPOINT_DIR / "preference-dpo"
).to(device)

freeze_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR / "partial-freeze"
).to(device)
n_layers = len(freeze_model.model.layers)
unfreeze_from = n_layers - max(2, n_layers // 4)

for parameter in freeze_model.parameters():
    parameter.requires_grad = False
for layer in freeze_model.model.layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True

# Llama ties lm_head to token embeddings; keep both frozen in the partial strategy.
assert freeze_model.lm_head.weight is freeze_model.model.embed_tokens.weight
assert not freeze_model.model.embed_tokens.weight.requires_grad

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = PeftModel.from_pretrained(
    lora_pt_base, CHECKPOINT_DIR / "peft-lora"
).to(device)

adapter_models = {
    "instruction LoRA": instruct_lora_model,
    "DPO policy": policy_model,
    "continued-pretraining LoRA": lora_pt_model,
}
expected_targets = set(LORA_TARGET_MODULES)
for adapter_name, adapter_model in adapter_models.items():
    configured_targets = {
        target
        for peft_config in adapter_model.peft_config.values()
        for target in peft_config.target_modules
    }
    if configured_targets != expected_targets:
        raise ValueError(
            f"{adapter_name} targets {sorted(configured_targets)}, expected "
            f"{sorted(expected_targets)}. Rerun Parts 1-2 with the SmolLM2 recipe."
        )

for model in (
    non_instruct_ckpt,
    instruct_lora_model,
    policy_model,
    freeze_model,
    lora_pt_model,
):
    model.eval()

trainable_partial_names = [
    name for name, parameter in freeze_model.named_parameters() if parameter.requires_grad
]
assert any(
    name.startswith(f"model.layers.{unfreeze_from}.")
    for name in trainable_partial_names
), "Expected SmolLM2 trailing-layer parameters were not found"

print("Reloaded all six candidates (baseline + 5 fine-tuned).")
print(f"SmolLM2 layers/hidden size: {n_layers} / {freeze_model.config.hidden_size}")

---

## Compare Candidates Without Mixing the Questions

| Candidate | What it practiced | Where its update lives | Intended improvement |
| --- | --- | --- | --- |
| Baseline | Original pretraining | No Riverside update | Control |
| Continued pretraining | Riverside next-token prediction | Full FT | Less-generic catalog prose |
| Partial-freeze continuation | Same prose task | Selected late layers | Lower trainable-state cost |
| LoRA continuation | Same prose task | Low-rank adapters | Small swappable update |
| SFT assistant | Request/response demonstrations | LoRA | Instruction compliance |
| DPO assistant | Chosen/rejected pairs | Continued SFT adapter | Better choices among valid answers |

The runs also used different data and settings. Their outputs can reveal interesting differences, but cannot tell us that one parameter strategy caused them.

## Begin with One Fixed Riverside Continuation

| Role | Text |
| --- | --- |
| Prompt | `Aria Voss checked the Meridian's Promise status panel and` |
| Riverside continuation | ` opened the Keeper's maintenance logs` |
| Generic alternative | ` looked at the screen` |

Ask one narrow question: did continued pretraining make the Riverside continuation less surprising than before, and did it change more than the generic alternative?

> **Predict:** Which will be harder to judge consistently: generic prose, instruction failure, or editorial preference?

### Why One Generated Output Is Not Enough

Generation chooses one token and then continues from that choice, so an early lucky or unlucky token changes everything after it. To compare the models themselves, keep the prompt and continuation fixed, inspect the probability of each actual next token, then repeat across more Riverside text.

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained("./checkpoints/...")` reloads a saved
> fine-tuned checkpoint from disk into a fresh `torch.nn.Module`, then `.to(device)` places it on
> GPU/CPU before the loop below calls the `generate()` helper defined earlier on each model in turn.
> **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(path)` loads the same checkpoint
> format into a `tf.keras.Model`; TensorFlow doesn't need an explicit `.to(device)` call since device
> placement is handled by default device scoping (or `tf.distribute` for multi-device setups) instead.


In [ ]:
# Compare every candidate on shared catalog prompts.
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("QUALITATIVE CANDIDATE EXAMPLES - SAME PROMPTS, ONE SAMPLE EACH")
print("=" * 80)

for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f'\nPrompt ({prompt_name}): "{prompt}"')
    print("-" * 80)

    for model_index, (model_name, model) in enumerate(models_to_test.items()):
        sample_seed = 42 + model_index
        torch.manual_seed(sample_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(sample_seed)

        uses_instruction_template = "Instruction" in model_name or "Preference" in model_name
        effective_prompt = instruction_prompt(prompt) if uses_instruction_template else prompt
        output = generate(
            model,
            effective_prompt,
            max_new_tokens=60,
            use_instruction_template=uses_instruction_template,
        )
        output_display = output[:120] + "..." if len(output) > 120 else output

        format_note = " + instruction template" if uses_instruction_template else ""
        print(f"\n[{model_name}] seed={sample_seed}{format_note}")
        print(f"  Output: {output_display}")

print("\n" + "=" * 80)
print("READ THESE AS EXAMPLES:")
print("1. Catalog language: are names and setting details specific rather than generic?")
print("2. Task behavior: does the model continue prose or answer an instruction?")
print("3. Stability: a claim requires repeated prompts/samples and a scoring rubric.")
print("=" * 80)

### Inspect One Complete Output

The prompt matrix gives breadth but truncates each sample. Before leaving qualitative evidence, inspect one complete output from every candidate and expose the exact input format.

This matters because SFT and DPO learned through the shared instruction template, while continuation models learned from plain prose. Sending every model the same raw string would test prompt mismatch as well as model behavior.

Use the same three lenses:

| Lens | Concrete sign to look for | Common false positive |
| --- | --- | --- |
| Domain language | Story-specific entities or relationships used coherently | Repeating a name copied from the prompt |
| Task behavior | Direct answer or bounded continuation in the requested format | Fluent prose that ignores the instruction |
| Preference signal | A repeatable difference between SFT and DPO | One nicer sample caused by decoding randomness |

Even the complete outputs remain sampled observations. The next step removes decoding randomness by scoring the same fixed phrases under both models.

In [ ]:
# Instruction and preference candidates use the explicit instruction contract from Parts 1-2.
instruct_prompt = instruction_prompt(PROMPT)
print(f"Shared prompt (continuation models) : {PROMPT!r}")
print(f"Shared user request (instruction models): {instruct_prompt!r}")
print()

print("=== Baseline (no fine-tuning) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(base_model, PROMPT)}")
print()

print("=== Non-instructional continued pretraining (full fine-tune) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(non_instruct_ckpt, PROMPT)}")
print()

print("=== Instruction-tuned (LoRA) ===")
print(f"  Input : instruction template over {instruct_prompt!r}")
print(
    f"  Output: {generate(instruct_lora_model, instruct_prompt, use_instruction_template=True)}"
)
print()

print("=== Preference-aligned (DPO on the instruction-tuned adapter) ===")
print(f"  Input : instruction template over {instruct_prompt!r}")
print(f"  Output: {generate(policy_model, instruct_prompt, use_instruction_template=True)}")
print()

print("=== Partial fine-tuning (layer freezing) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(freeze_model, PROMPT)}")
print()

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(lora_pt_model, PROMPT)}")
print()

### From One Sample to Fixed Text

Sampled outputs mix model probabilities with decoder choices: once one token differs, every later token sees a different history. Fix both the manuscript prompt and continuation so each candidate reads the same text. Softmax supplies a shared probability budget over next-token alternatives; the useful value is the probability assigned to the token that actually comes next.

### Teacher Forcing: Keep Every Candidate on the Same Text

When scoring token $w_t$, supply the actual preceding tokens $w_{<t}$ and gather the model's probability for the actual next token. Repeat for every target position.

This is **teacher forcing**. It prevents candidate histories from forking, so a probability difference belongs to the models rather than an earlier sampled choice.

The interpretation remains narrow: high actual-token probability means “expected under this model after this context,” not true, safe, or preferred.

### Score One Fixed Continuation

To keep the arithmetic readable, consider the first two illustrative target tokens. The real tokenizer may divide the text differently; the next code cell prints its actual tokenization and scores every resulting token.

| Scoring step | Context supplied to model | Actual target token | Probability gathered |
| ---: | --- | --- | ---: |
| 1 | `... status panel and` | ` opened` | $0.50$ |
| 2 | `... status panel and opened` | ` the` | $0.20$ |

The probability assigned to this two-token prefix is the product of its conditional probabilities:

$$
p(\text{ opened the}\mid\text{walking prompt})
=0.50\times0.20=0.10.
$$

The full target adds probabilities for `Keeper`, `'s`, `maintenance`, `logs`, or whatever token pieces the tokenizer actually produces. Products across many tokens quickly become tiny. Logs turn multiplication into addition:

$$
\log 0.50+\log 0.20=-0.693-1.609=-2.303.
$$

Divide by the two predicted tokens to place this prefix on a per-token scale:

$$
\text{mean log-probability}
=\frac{-2.303}{2}=-1.151\ \text{nats/token}.
$$

For the complete continuation, average the log-probabilities assigned to every actual continuation token. A higher mean log-probability means that fixed continuation was less surprising to that model. Negative log-likelihood (NLL) flips the sign, so lower NLL means the same thing.

The continuation is reference text supplied for scoring; the metric does not decide that it is the uniquely correct answer.

### Compare the Model Changes

A mean log-probability is one score for **one model reading one fixed continuation**. The baseline comparison appears only when we put the same continuation under both models.

The values below are illustrative. The code calculates the same columns from the real checkpoints.

| Fixed continuation after the same prompt | Base model score | Adapted model score | Change after adaptation |
| --- | ---: | ---: | ---: |
| Riverside target: ` opened the Keeper's maintenance logs` | $-5.0$ | $-3.0$ | $+2.0$ |
| Generic control: ` looked at the screen` | $-2.0$ | $-1.8$ | $+0.2$ |

Read the matrix in this order:

1. **Stay within one row first.** For the Riverside phrase, the score moved from $-5.0$ to $-3.0$, so this fixed phrase became less surprising after adaptation.
2. **Repeat the same before/after comparison for the generic phrase.** It also became less surprising, but only by $+0.2$.
3. **Compare the two changes, not the raw scores.** The Riverside phrase gained $+2.0$, while the generic phrase gained $+0.2$. The Riverside phrase therefore gained $+1.8$ more.

Do **not** compare the raw adapted scores $-3.0$ and $-1.8$ directly. The generic phrase may simply have been easier for both models before training. The useful question is whether adaptation changed each phrase differently from its own starting point.

The short labels for those two subtractions are:

- phrase shift: $\Delta(c)=\text{adapted score for }c-\text{base score for }c$;
- selectivity contrast: $\Delta(\text{Riverside phrase})-\Delta(\text{generic phrase})$.

For the worked matrix, the selectivity contrast is $(+2.0)-(+0.2)=+1.8$. For this pair, the positive result suggests that adaptation favored the Riverside-specific phrase more than the plausible generic alternative.

The **Riverside target** is a deliberately domain-specific probe phrase, not “the one correct answer.” The **generic control** is a plausible continuation with no Riverside-specific content. It tells us whether the score change was broad rather than specifically tied to the Riverside material.

This still covers only the chosen phrases. It does not establish generalization, overall writing quality, factual correctness, or instruction following.

### Predict Before Measuring

Continued pretraining should improve the Riverside continuation more than the generic alternative. The next cell calculates both changes and compares them.

A positive difference says this one Riverside phrase benefited more. Repeating the same check across many passages tells us whether that pattern is broader.

> **PyTorch → Keras:** `model.eval()` switches dropout/batch-norm-style layers to inference behavior;
> `torch.no_grad()` disables gradient tracking for the forward pass below; calling `model(**inputs)`
> runs a forward pass and returns `outputs.logits` (raw scores), and `F.softmax(logits, dim=-1)`
> (from `torch.nn.functional`) converts those logits into a probability distribution over the vocabulary.
> **Keras/TF equivalent:** Keras layers infer train/inference behavior automatically (or via a
> `training=False` argument) instead of an explicit `.eval()` call, and there's no separate "no_grad"
> context since plain forward calls outside a `GradientTape` don't track gradients; the softmax step is
> `tf.nn.softmax(logits, axis=-1)` -- same idea, `axis` instead of `dim`.


In [ ]:
# Trace one fixed sentence token by token, then compare complete continuation scores.
import math

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

prompt_for_analysis = "Aria Voss checked the Meridian's Promise status panel and"
walking_target_label = "Keeper's maintenance logs"
generic_control_label = "looked at the screen"

# The first phrase is the walking target; the remaining phrases test selectivity.
candidate_phrases = {
    walking_target_label: " opened the Keeper's maintenance logs",
    "quantum fold drive": " checked the quantum fold drive",
    "containment-field anomaly": " detected a containment-field anomaly",
    generic_control_label: " looked at the screen",
    "said nothing": " said nothing",
    "went back to work": " went back to work",
}
domain_labels = set(list(candidate_phrases)[:3])


def continuation_logprob(model, prompt, continuation):
    """Return sequence summaries and the actual-token trace for one continuation."""
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = tokenizer(
        prompt + continuation, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + continuation")

    model.eval()
    with torch.no_grad():
        logits = model(full_ids).logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    # Position prompt_length-1 predicts the first continuation token.
    continuation_ids = full_ids[:, prompt_length:]
    continuation_log_probs = log_probs[
        0, prompt_length - 1 : full_ids.shape[1] - 1
    ].gather(1, continuation_ids[0].unsqueeze(1)).squeeze(1)

    token_ids = continuation_ids[0].tolist()
    token_log_probs = continuation_log_probs.tolist()
    token_pieces = tokenizer.convert_ids_to_tokens(token_ids)
    trace = [
        {
            "token_id": token_id,
            "token": token_piece,
            "probability": math.exp(token_log_probability),
            "log_probability": token_log_probability,
            "surprise": -token_log_probability,
        }
        for token_id, token_piece, token_log_probability in zip(
            token_ids, token_pieces, token_log_probs
        )
    ]

    return {
        "mean": continuation_log_probs.mean().item(),
        "sum": continuation_log_probs.sum().item(),
        "tokens": continuation_ids.shape[1],
        "trace": trace,
    }


phrase_results = []
walking_traces = None
for label, phrase in candidate_phrases.items():
    baseline_score = continuation_logprob(base_model, prompt_for_analysis, phrase)
    finetuned_score = continuation_logprob(
        non_instruct_ckpt, prompt_for_analysis, phrase
    )
    if label == walking_target_label:
        walking_traces = {
            "baseline": baseline_score["trace"],
            "finetuned": finetuned_score["trace"],
        }
    phrase_results.append(
        {
            "label": label,
            "kind": "catalog" if label in domain_labels else "generic control",
            "tokens": finetuned_score["tokens"],
            "baseline": baseline_score["mean"],
            "finetuned": finetuned_score["mean"],
            "delta": finetuned_score["mean"] - baseline_score["mean"],
        }
    )

assert walking_traces is not None
assert [item["token_id"] for item in walking_traces["baseline"]] == [
    item["token_id"] for item in walking_traces["finetuned"]
]

print("=== Walking problem: actual next-token trace ===")
print(f"Prompt: {prompt_for_analysis!r}")
print(f"Fixed target: {candidate_phrases[walking_target_label]!r}\n")
print(
    f"{'Step':>4} {'Tokenizer piece':22} {'Base p':>10} {'Adapted p':>10} "
    f"{'Base surprise':>14} {'Adapted surprise':>17}"
)
print("-" * 92)
for step, (baseline_token, finetuned_token) in enumerate(
    zip(walking_traces["baseline"], walking_traces["finetuned"]), start=1
):
    print(
        f"{step:>4} {baseline_token['token']!r:22} "
        f"{baseline_token['probability']:>10.5f} "
        f"{finetuned_token['probability']:>10.5f} "
        f"{baseline_token['surprise']:>14.3f} "
        f"{finetuned_token['surprise']:>17.3f}"
    )
print(
    "\nRead one row at a time: both models see the same true prefix; lower surprise "
    "means the model assigned more probability to that actual next token."
)

print("\n=== Complete fixed-continuation scores ===")
print(
    f"{'Phrase':30} {'Type':16} {'Tok':>3} {'Base':>9} "
    f"{'Adapted':>11} {'Adapted - base':>15}"
)
print("-" * 94)
for row in phrase_results:
    print(
        f"{row['label']:30} {row['kind']:16} {row['tokens']:>3} "
        f"{row['baseline']:>9.3f} {row['finetuned']:>11.3f} {row['delta']:>+15.3f}"
    )

labels = [row["label"] for row in phrase_results]
baseline_values = [row["baseline"] for row in phrase_results]
finetuned_values = [row["finetuned"] for row in phrase_results]
deltas = [row["delta"] for row in phrase_results]
positions = np.arange(len(labels))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.35, 1]})

axes[0].barh(
    positions - width / 2,
    baseline_values,
    height=width,
    label="Baseline",
    color="#4C78A8",
)
axes[0].barh(
    positions + width / 2,
    finetuned_values,
    height=width,
    label="Continued pretraining",
    color="#E45756",
)
axes[0].set_yticks(positions)
axes[0].set_yticklabels(labels)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean log-probability per token (higher = less surprising)")
axes[0].set_title("One-Model Scores for Fixed Text")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.25)

delta_colors = ["#2A9D8F" if value >= 0 else "#D1495B" for value in deltas]
axes[1].barh(positions, deltas, color=delta_colors)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Adapted minus base (nats/token)")
axes[1].set_title("Baseline Comparison for Each Phrase")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

catalog_deltas = [row["delta"] for row in phrase_results if row["kind"] == "catalog"]
control_deltas = [
    row["delta"] for row in phrase_results if row["kind"] == "generic control"
]
walking_delta = next(
    row["delta"] for row in phrase_results if row["label"] == walking_target_label
)
generic_control_delta = next(
    row["delta"] for row in phrase_results if row["label"] == generic_control_label
)
walking_selectivity_delta = walking_delta - generic_control_delta

print("\n=== Nested comparisons ===")
print(f"Walking target, adapted - base:       {walking_delta:+.3f} nats/token")
print(
    f"Named generic control, adapted - base: {generic_control_delta:+.3f} nats/token"
)
print(
    f"Selectivity contrast (target - control): {walking_selectivity_delta:+.3f} nats/token"
)
print(f"Mean catalog shift:                    {np.mean(catalog_deltas):+.3f} nats/token")
print(f"Mean all-generic-controls shift:        {np.mean(control_deltas):+.3f} nats/token")
print(
    "Interpretation: each phrase shift compares adapted with base. The selectivity "
    "contrast then asks whether the walking target shifted more than the named generic control."
)
print(
    "A positive selectivity contrast suggests a local Riverside-specific pattern; "
    "it is not a general quality or corpus-level result."
)

### From One Sentence to the Corpus

One favorable phrase may be luck. Carry the same calculation from one token to many Riverside passages:

> all next-token probabilities -> probability of the token that occurred -> surprise -> average surprise -> perplexity

| View | Plain-language role |
| --- | --- |
| Vocabulary distribution | Every token the model considered next |
| Actual-token probability | How much probability reached the token that occurred |
| Surprise | A larger penalty when that token was considered unlikely |
| Mean NLL | Average penalty across a phrase or corpus |
| Perplexity | The same average on a more readable scale |

After the Riverside prompt, imagine the adapted model moves probability from generic alternatives toward ` opened`. The corpus check asks whether it repeatedly does that for real Riverside text, not only this hand-picked position.

### Why Perplexity Appears

We need one number that makes an almost-impossible observed token hurt more than an ordinary miss. Define **surprise**:

$$
\text{surprise}=-\ln p(\text{actual token}).
$$

| Probability | Surprise | Reading |
| ---: | ---: | --- |
| $0.80$ | $0.22$ | Strongly expected |
| $0.20$ | $1.61$ | Many alternatives remained |
| $0.001$ | $6.91$ | Almost ruled out |

Average surprise gives mean NLL. Exponentiating that average gives **perplexity**, which expresses the same result on a friendlier scale. It does not judge truth, instruction following, or writing quality; it only summarizes how expected the supplied text was.

### Repeat the Check Across More Passages

The selected continuation may be unusually favorable. Repeat the same teacher-forced scoring across many Riverside passages:

> Does the adapted model generally find the words that actually occur less surprising than the base model does?

This notebook's shared corpus check is useful for seeing how the calculation scales up. Some candidates may already have seen parts of that text, so do not use its bars to choose a production model.

> **PyTorch → Keras:** passing `labels=enc["input_ids"]` into the model's forward call makes the
> Hugging Face model compute cross-entropy loss internally and return it as `out.loss` (a scalar
> tensor); `torch.no_grad()` skips gradient tracking since this is evaluation-only, and `.item()`
> pulls the plain Python float out of that 0-d tensor, which `math.exp(loss)` then turns into
> perplexity. **Keras/TF equivalent:** the TF counterpart model supports the same `labels=` convenience
> (`model(enc, labels=...)`), while plain Keras code would instead call
> `tf.keras.losses.SparseCategoricalCrossentropy()(y_true, logits)` and use `.numpy()` in place of
> `.item()` to extract the scalar.


In [ ]:
# Reuse the notebook and repository paths resolved during setup.
CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():
    fallback_content_dir = REPO_ROOT / "learning" / "genai" / "03-llm-finetuning" / "content"
    if fallback_content_dir.exists():
        CONTENT_DIR = fallback_content_dir

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}

if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Riverside content directory not found at {CONTENT_DIR.absolute()}. "
        "Run this notebook from the repository workspace."
    )

print(f"Corpus evaluation setup: {CONTENT_DIR.absolute()} ({len(NOVELS)} novels)")

In [ ]:
import math


# Use the same later-chapter sample for every candidate. This is a shared probe, not a clean holdout,
# because upstream training coverage differs and full FT saw some of these files.
def load_corpus_probe(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []
    source_files = []

    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        probe_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        source_files.extend(probe_files)

        for path in probe_files:
            text = path.read_text(encoding="utf-8")
            for paragraph in text.split("\n\n"):
                paragraph = paragraph.strip().replace("\n", " ")
                if len(paragraph) >= min_len:
                    paragraphs.append(paragraph)

    return paragraphs, source_files


probe_paragraphs, probe_files = load_corpus_probe()
print(
    f"Shared corpus probe: {len(probe_paragraphs)} paragraphs from "
    f"{len(probe_files)} later-chapter files."
)
print(
    "WARNING: this is descriptive, not held out. Upstream candidates used different "
    "training files, and full fine-tuning saw some probe chapters."
)


# Aggregate negative log-likelihood by evaluated token rather than averaging paragraph means.
def compute_corpus_probe(model, paragraphs, max_length=64):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            output = model(**encoded, labels=encoded["input_ids"])

            # Causal loss predicts tokens 2..N from tokens 1..N-1.
            valid_tokens = int(encoded["attention_mask"][:, 1:].sum().item())
            total_nll += output.loss.item() * valid_tokens
            total_tokens += valid_tokens

    mean_nll = total_nll / total_tokens
    return {
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "tokens": total_tokens,
    }


models_for_probe = {
    "Baseline (no fine-tuning)": base_model,
    "Full fine-tuning": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial freezing": freeze_model,
    "LoRA continued pretraining": lora_pt_model,
}

print(f"\n{'=' * 72}\nShared corpus-probe perplexity (descriptive only):\n{'=' * 72}")
corpus_probe_results = {}
for name, model in models_for_probe.items():
    result = compute_corpus_probe(model, probe_paragraphs)
    corpus_probe_results[name] = result
    print(
        f"  {name:<32} NLL={result['mean_nll']:6.3f}  "
        f"PPL={result['perplexity']:8.1f}  tokens={result['tokens']:,}"
    )
print(f"{'=' * 72}")

probe_ranking = sorted(
    corpus_probe_results.items(), key=lambda item: item[1]["perplexity"]
)
names = [name for name, _ in probe_ranking]
perplexities = [result["perplexity"] for _, result in probe_ranking]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names[::-1], perplexities[::-1], color="#4C78A8")
ax.set_xlabel("Corpus-probe perplexity (lower = better fit to this sample)")
ax.set_title(
    "Shared Later-Chapter Corpus Probe\n"
    "Descriptive only: training exposure differs across candidates",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

lowest_name, lowest_result = probe_ranking[0]
print(
    f"Lowest value on this contaminated probe: {lowest_name!r} "
    f"(PPL={lowest_result['perplexity']:.1f})."
)
print(
    "Do not promote that candidate from this ranking. A valid selection requires a split "
    "created before matched training plus workload-specific task metrics."
)

## What the Prose Checks Told Us

The comparison grew in three steps:

1. generated text showed visible behavior;
2. fixed text separated model probabilities from decoding choices;
3. corpus perplexity checked more prose at once.

That answers whether the model fits Riverside prose better. It does not answer whether an assistant obeys a request or whether editors prefer its answer, so those sections count different things.

## Change the Question, Change What You Count

| Training goal | Riverside asks | What decides it |
| --- | --- | --- |
| Continued pretraining | Did held-out prose become easier for the model to predict? | Corpus NLL/perplexity plus general-language checks |
| SFT | Did the assistant satisfy the request completely? | Contract pass rate |
| DPO | Do editors prefer DPO to the accepted SFT answer? | Blinded wins, losses, and ties |
| Parameter strategy | Did a cheaper update preserve the behavior? | The same behavior check plus measured cost |

The prose number has done its job. Keep the Riverside scenario, but count the behavior that the new training objective was intended to change.

## SFT Evaluation: Count Complete Contracts, Not Nice Sentences

Suppose Riverside requires exactly one sentence, a clean stop, and no unsupported manuscript fact. A response that satisfies two of those three rules is still unusable for that case.

That creates the metric naturally:

1. evaluate each required rule;
2. mark the case as passed only when **all** rules pass;
3. repeat across held-out requests;
4. count the fraction of complete passes.

$$
\text{instruction pass rate}=\frac{\text{cases satisfying every required rule}}{N}.
$$

The next code cell builds five visible case records so the aggregation can be inspected rather than accepted as notation. Deterministic code should own explicit format rules; qualified review handles semantic support or usefulness when rules cannot.

---

## DPO Evaluation: Count Repeated Choices Between Valid Answers

Once both SFT and DPO satisfy the contract, the remaining question is comparative: which answer would an editor keep? One attractive sample is not a preference result.

For each held-out request:

1. generate SFT and DPO answers under matched decoding;
2. reject answers that fail the SFT contract;
3. hide model identity and randomize A/B order;
4. record `DPO win`, `SFT win`, or `tie`.

Across repeated choices, ties receive half credit:

$$
\text{DPO win rate}=\frac{W+0.5T}{W+L+T}.
$$

The executable example shows how individual judgments become the summary. The number is relative to the named SFT reference, not an absolute percentage of answer quality.

---

## Parameter Strategy: Quality First, Cost Second

Full fine-tuning, freezing, LoRA, and QLoRA change where an update lives, not the user goal. Hold objective, data, budget, and evaluation fixed. Eliminate candidates that miss the behavior gate, then compare the resource constraints Riverside actually has. The same code cell prints the real update scopes of the loaded candidates without treating them as quality scores.

In [ ]:
# Worked aggregation examples: visible records first, summary metrics second.
worked_sft_cases = [
    {"case": "one-sentence continuation", "one_sentence": True, "clean_stop": True, "source_supported": True},
    {"case": "format drift", "one_sentence": False, "clean_stop": True, "source_supported": True},
    {"case": "unsupported detail", "one_sentence": True, "clean_stop": True, "source_supported": False},
    {"case": "extra dialogue turn", "one_sentence": True, "clean_stop": False, "source_supported": True},
    {"case": "second valid continuation", "one_sentence": True, "clean_stop": True, "source_supported": True},
]
required_rules = ("one_sentence", "clean_stop", "source_supported")

print("WORKED SFT CONTRACT RECORDS")
print(f"{'Case':32} {'sentence':>9} {'stop':>7} {'support':>9} {'case pass':>10}")
print("-" * 73)
for record in worked_sft_cases:
    record["contract_pass"] = all(record[rule] for rule in required_rules)
    print(
        f"{record['case']:32} "
        f"{str(record['one_sentence']):>9} "
        f"{str(record['clean_stop']):>7} "
        f"{str(record['source_supported']):>9} "
        f"{str(record['contract_pass']):>10}"
    )

complete_passes = sum(record["contract_pass"] for record in worked_sft_cases)
instruction_pass_rate = complete_passes / len(worked_sft_cases)
print(
    f"\nInstruction pass rate = complete passes / cases "
    f"= {complete_passes} / {len(worked_sft_cases)} = {instruction_pass_rate:.1%}"
)
print("One failed required rule fails the case; partial credit would hide an unusable response.\n")

worked_preference_judgments = [
    "DPO win",
    "SFT win",
    "DPO win",
    "tie",
    "DPO win",
    "tie",
    "SFT win",
    "DPO win",
]
dpo_wins = worked_preference_judgments.count("DPO win")
sft_wins = worked_preference_judgments.count("SFT win")
ties = worked_preference_judgments.count("tie")
dpo_win_rate = (dpo_wins + 0.5 * ties) / len(worked_preference_judgments)

print("WORKED BLINDED PREFERENCE RECORDS")
print("Judgments:", ", ".join(worked_preference_judgments))
print(
    f"DPO win rate = (wins + 0.5 * ties) / judgments "
    f"= ({dpo_wins} + 0.5 * {ties}) / {len(worked_preference_judgments)} "
    f"= {dpo_win_rate:.1%}"
)
print(
    f"SFT wins remain visible ({sft_wins}); the summary compares DPO with this named SFT reference.\n"
)

# Add real update scopes when the earlier candidate-loading cells have run.
required_models = ("base_model", "non_instruct_ckpt", "freeze_model", "instruct_lora_model")
if all(model_name in globals() for model_name in required_models):
    total_params = sum(parameter.numel() for parameter in base_model.parameters())
    strategy_counts = {
        "Full fine-tuning": sum(parameter.numel() for parameter in non_instruct_ckpt.parameters()),
        "Partial freezing": sum(
            parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
        ),
        "LoRA matrices": sum(
            parameter.numel()
            for name, parameter in instruct_lora_model.named_parameters()
            if ".lora_" in name
        ),
    }

    print("LOADED CANDIDATE UPDATE SCOPES")
    for strategy, count in strategy_counts.items():
        print(f"  {strategy:<20} {count:>12,} parameters ({count / total_params:>7.2%})")
    print("These counts describe update scope; matched behavior evidence decides sufficiency.")
else:
    print("SKIP update-scope appendix: run the earlier candidate-loading cells first.")

## Matched Comparison: Change One Thing

To learn whether LoRA or full fine-tuning caused a difference, train both on the same Riverside request with the same base model, ordered examples, split, instruction template, optimizer policy, token budget, evaluation code, and seeds. Change only how the update is stored.

1. Check instruction following, source support, and safety first.
2. Inspect difficult requests instead of trusting only the average.
3. Compare memory, training time, artifact size, latency, and cost only among candidates that pass the behavior checks.

Choose LoRA when it keeps the required behavior and reduces a real burden. Choose full fine-tuning only when its repeatable improvement matters enough to pay for.

![Two Riverside experiment workbenches: a tangled comparison where data, objective, budget, and seed all change, beside a clean comparison where identical inputs feed full fine-tuning and LoRA and only the update strategy differs](images/controlled-comparison-vs-confounding.png)

The existing checkpoints were trained with different data and settings, so they locate the comparison Riverside still needs; they do not answer it.

The following prompt test asks a separate question: can supplied context provide a current fact without changing model weights?

### Put the Prompting Question to a Fairer Test

“Who is Aria Voss?” alone mixes two questions: does the model know a private fact, and does it know how to answer directly? A three-way observation separates them:

1. **Base model, no Riverside context:** can only use knowledge already present in its weights.
2. **Base model, fact supplied in the prompt:** tests whether prompting can provide current evidence without changing weights.
3. **SFT adapter, no supplied fact:** tests the learned answer contract, but still cannot guarantee that an unsupported catalog claim is true.

This remains one example, not an evaluation suite. Its purpose is to clarify the roles of prompting and fine-tuning before Riverside chooses a production architecture.

In [ ]:
# One observation that separates unavailable facts from answer behavior.
zero_shot_prompt = "Who is Aria Voss?"
provided_context = (
    "Riverside context: Aria Voss serves aboard the Meridian's Promise and "
    "investigates a signal counting itself out in prime numbers.\n\n"
    "Based only on that context, who is Aria Voss?"
)

print("=== Base model, no Riverside context ===")
print(generate(base_model, zero_shot_prompt, use_instruction_template=True), "\n")

print("=== Base model, Riverside fact supplied in the prompt ===")
print(generate(base_model, provided_context, use_instruction_template=True), "\n")

print("=== SFT adapter, no Riverside context supplied ===")
print(generate(instruct_lora_model, zero_shot_prompt, use_instruction_template=True))

print(
    "\nRead this as a role check: supplied context can provide current facts; "
    "SFT can change response behavior; neither single output proves reliable factual recall."
)

## Put It Together: One Riverside Scenario

> **Scenario:** A character aboard the *Meridian's Promise* discovers an anomaly. Which candidate looks useful, and what should Riverside check next?

The next worksheet applies the notebook's progression once from visible outputs to a workload decision. It does not force every candidate into one score.

### What the Worksheet Does

| Step | What you do | What it tells you |
| --- | --- | --- |
| 1. Generate | Read fresh outputs under the input format each model learned | Which differences deserve a closer look |
| 2. Inspect | Use the same rubric for specificity, role, coherence, and support risk | Why an output looked useful or failed |
| 3. Fix phrases | Compare the same continuations under the prose models | Whether Riverside phrases shifted more than generic ones |
| 4. Broaden | Repeat on a small same-novel corpus | Whether the prose pattern appears beyond one phrase |

SFT and DPO belong in the output inspection, but prose likelihood does not rank them. Their own sections already showed what to count instead.

### Riverside's Editing Assistant

SFT-LoRA is the candidate to test because it practiced the editing request. DPO matters only after SFT follows the rules reliably and editors repeatedly prefer the DPO answers. Current manuscript facts must come from supplied or retrieved passages.

### Before You Run

The next four cells form one worksheet. Change `WALKING_SAMPLE_SEED` to see whether your impression survives another generation, and fill the rubric before reading the combined table.

Use fixed-phrase scores and prose perplexity only for the baseline and continued-pretraining models.

In [ ]:
# Walking example, step 1: generate fresh outputs from every candidate.
WALKING_SAMPLE_SEED = 2026
WALKING_PROMPT = "Aria Voss checked the Meridian's Promise status panel and"

walking_candidates = {
    "Baseline": {
        "model": base_model,
        "objective": "Original pretraining",
        "use_instruction_template": False,
    },
    "Full-FT continuation": {
        "model": non_instruct_ckpt,
        "objective": "Continued pretraining",
        "use_instruction_template": False,
    },
    "Partial-freeze continuation": {
        "model": freeze_model,
        "objective": "Continued pretraining",
        "use_instruction_template": False,
    },
    "LoRA continuation": {
        "model": lora_pt_model,
        "objective": "Continued pretraining",
        "use_instruction_template": False,
    },
    "SFT LoRA": {
        "model": instruct_lora_model,
        "objective": "Supervised instruction tuning",
        "use_instruction_template": True,
    },
    "DPO adapter": {
        "model": policy_model,
        "objective": "Preference optimization after SFT",
        "use_instruction_template": True,
    },
}

walking_outputs = {}
for candidate_name, candidate in walking_candidates.items():
    # Reuse one seed so differences are not caused by assigning easier random streams to some models.
    torch.manual_seed(WALKING_SAMPLE_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(WALKING_SAMPLE_SEED)

    uses_instruction_template = candidate["use_instruction_template"]
    effective_prompt = (
        instruction_prompt(WALKING_PROMPT) if uses_instruction_template else WALKING_PROMPT
    )
    completion = generate(
        candidate["model"],
        effective_prompt,
        max_new_tokens=80,
        use_instruction_template=uses_instruction_template,
    )
    walking_outputs[candidate_name] = {
        "objective": candidate["objective"],
        "input_contract": (
            "explicit instruction template" if uses_instruction_template else "plain continuation"
        ),
        "effective_prompt": effective_prompt,
        "completion": completion,
    }

    print("=" * 88)
    print(f"Candidate      : {candidate_name}")
    print(f"Objective      : {candidate['objective']}")
    print(f"Input contract : {walking_outputs[candidate_name]['input_contract']}")
    print(f"Effective input: {effective_prompt!r}")
    print(f"Output         : {completion}")

print("\nRerun with a different WALKING_SAMPLE_SEED to test whether an impression persists.")

In [ ]:
# Walking example, step 2: inspect the live outputs and record manual observations.
WALKING_RUBRIC = {
    "catalog_specificity": "0=generic, 1=mentions Riverside details, 2=uses details coherently",
    "role_fulfillment": "0=wrong behavior, 1=partly fulfills its role, 2=clear bounded continuation/answer",
    "coherence": "0=contradictory, 1=mostly coherent, 2=coherent across the full completion",
    "unsupported_claim_risk": "0=high risk, 1=uncertain, 2=no unsupported catalog claim observed",
}

# Replace None with 0, 1, or 2 after reading the generated outputs above.
# Keep notes concrete: quote the phrase that earned or lost a point.
walking_manual_scores = {
    candidate_name: {
        "catalog_specificity": None,
        "role_fulfillment": None,
        "coherence": None,
        "unsupported_claim_risk": None,
        "notes": "",
    }
    for candidate_name in walking_outputs
}

# After comparing the two instruction candidates, set this to "SFT LoRA", "DPO adapter", or "tie".
# One choice is an observation for this sample, not a preference win-rate estimate.
walking_preference_choice = None

print("MANUAL INSPECTION RUBRIC")
print("=" * 88)
for dimension, definition in WALKING_RUBRIC.items():
    print(f"{dimension:24} {definition}")

print("\nOUTPUT WORKSHEET")
print("=" * 88)
for candidate_name, output_record in walking_outputs.items():
    print(f"\n[{candidate_name}] {output_record['input_contract']}")
    print(output_record["completion"])
    print("Scores:", walking_manual_scores[candidate_name])

print("\nEdit walking_manual_scores in this cell, rerun it, then continue.")
print("For a DPO claim, also record walking_preference_choice after comparing SFT LoRA with DPO adapter.")

In [ ]:
# Walking example, step 3: hold phrases fixed and compare nested probability shifts.
walking_continuation_models = {
    "Baseline": base_model,
    "Full-FT continuation": non_instruct_ckpt,
    "Partial-freeze continuation": freeze_model,
    "LoRA continuation": lora_pt_model,
}
walking_phrases = {
    "Keeper maintenance logs": ("catalog", " opened the Keeper's maintenance logs"),
    "quantum fold drive": ("catalog", " checked the quantum fold drive"),
    "containment anomaly": ("catalog", " detected a containment-field anomaly"),
    "looked at the screen": ("generic control", " looked at the screen"),
    "went back to work": ("generic control", " went back to work"),
}

walking_phrase_results = {}
for candidate_name, model in walking_continuation_models.items():
    candidate_results = {}
    for phrase_name, (phrase_type, continuation) in walking_phrases.items():
        score = continuation_logprob(model, WALKING_PROMPT, continuation)
        candidate_results[phrase_name] = {
            "type": phrase_type,
            "mean_logprob": score["mean"],
            "tokens": score["tokens"],
        }
    walking_phrase_results[candidate_name] = candidate_results

baseline_phrase_scores = walking_phrase_results["Baseline"]
walking_phrase_summary = {}
print(f"Fixed prompt: {WALKING_PROMPT!r}")
print(
    f"{'Candidate':28} {'Catalog Δ(A-B)':>15} {'Control Δ(A-B)':>15} "
    f"{'Selectivity Δ':>15}"
)
print("-" * 78)
for candidate_name, candidate_results in walking_phrase_results.items():
    catalog_deltas = [
        row["mean_logprob"] - baseline_phrase_scores[phrase_name]["mean_logprob"]
        for phrase_name, row in candidate_results.items()
        if row["type"] == "catalog"
    ]
    control_deltas = [
        row["mean_logprob"] - baseline_phrase_scores[phrase_name]["mean_logprob"]
        for phrase_name, row in candidate_results.items()
        if row["type"] == "generic control"
    ]
    mean_catalog_shift = float(np.mean(catalog_deltas))
    mean_control_shift = float(np.mean(control_deltas))
    selectivity_contrast = mean_catalog_shift - mean_control_shift
    walking_phrase_summary[candidate_name] = {
        "mean_catalog_shift": mean_catalog_shift,
        "mean_control_shift": mean_control_shift,
        "selectivity_contrast": selectivity_contrast,
    }
    print(
        f"{candidate_name:28} {mean_catalog_shift:>+15.3f} "
        f"{mean_control_shift:>+15.3f} {selectivity_contrast:>+15.3f}"
    )

print("\nΔ(A-B) means candidate score minus baseline-model score for the same phrase.")
print(
    "Selectivity Δ means mean catalog shift minus mean generic-control shift. "
    "Positive values suggest a local domain-selectivity pattern, not a quality claim."
)

In [ ]:
# Walking example, step 4: broaden to same-novel prose and assemble the comparison table.
WALKING_MAX_PROBE_PARAGRAPHS = 12
walking_probe_files = [
    path for path in probe_files if path.parent.name == NOVELS["scifi"]
]
walking_probe_paragraphs = []
for path in walking_probe_files:
    for paragraph in path.read_text(encoding="utf-8").split("\n\n"):
        paragraph = paragraph.strip().replace("\n", " ")
        if len(paragraph) >= 200:
            walking_probe_paragraphs.append(paragraph)
walking_probe_paragraphs = walking_probe_paragraphs[:WALKING_MAX_PROBE_PARAGRAPHS]

if not walking_probe_paragraphs:
    raise ValueError("No same-novel probe paragraphs were found; run the earlier corpus-loader cells first.")

walking_probe_results = {}
print(
    f"Same-novel descriptive probe: {len(walking_probe_paragraphs)} paragraphs "
    f"from {len(walking_probe_files)} file(s)"
)
for candidate_name, model in walking_continuation_models.items():
    result = compute_corpus_probe(model, walking_probe_paragraphs)
    walking_probe_results[candidate_name] = result
    print(
        f"  {candidate_name:28} "
        f"PPL={result['perplexity']:8.2f}  tokens={result['tokens']:,}"
    )


def walking_manual_total(candidate_scores):
    """Return a rubric total only after the reader has filled every score."""
    dimensions = [candidate_scores[name] for name in WALKING_RUBRIC]
    return None if any(value is None for value in dimensions) else sum(dimensions)


print("\nCOMBINED WALKING-EXAMPLE OBSERVATIONS")
print("=" * 132)
print(
    f"{'Candidate':28} {'Manual /8':>10} {'Catalog Δ(A-B)':>15} "
    f"{'Control Δ(A-B)':>15} {'Selectivity Δ':>15} {'Probe PPL':>12}"
)
print("-" * 132)
for candidate_name in walking_candidates:
    manual_total = walking_manual_total(walking_manual_scores[candidate_name])
    phrase_summary = walking_phrase_summary.get(candidate_name)
    probe_summary = walking_probe_results.get(candidate_name)
    manual_display = "pending" if manual_total is None else str(manual_total)
    catalog_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['mean_catalog_shift']:+.3f}"
    )
    control_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['mean_control_shift']:+.3f}"
    )
    selectivity_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['selectivity_contrast']:+.3f}"
    )
    probe_display = "n/a" if probe_summary is None else f"{probe_summary['perplexity']:.2f}"
    print(
        f"{candidate_name:28} {manual_display:>10} {catalog_display:>15} "
        f"{control_display:>15} {selectivity_display:>15} {probe_display:>12}"
    )

print("\nManual SFT-vs-DPO preference for this sample:", walking_preference_choice or "pending")
print("\nInterpretation prompts:")
print("1. Which live output suggested a pattern worth checking on a larger task suite?")
print("2. Did adapted-minus-base catalog shifts exceed generic-control shifts?")
print("3. Does same-novel perplexity point in the same direction, while remaining contaminated and non-causal?")
print("4. Which decision story applies, and what matched study or workload requirement is needed before promotion?")
print("\nDo not rank SFT or DPO by prose perplexity: their objectives require instruction and preference suites.")

---

## Four Riverside Decision Stories

The worksheet produced observations. These four short stories show how Riverside decides what those observations actually justify before applying formal release gates.

### 1. One Editor Changes One Vote: Mark It `REVIEW`

Eight editors compare SFT and DPO. DPO appears to win by one vote. One editor rereads the responses, notices that the preferred answer quietly violated the one-sentence contract, and changes the vote. The winner flips.

Riverside did not discover that the other model is better. It discovered that **one judgment can reverse the conclusion**. Mark the result `REVIEW`, preserve the disputed case, and gather more judgments under the same task definition. A narrow lead is an invitation to inspect, not a promotion decision.

### 2. Same Recipe, Different Seed: Luck or Strategy?

Riverside repeats the same training recipe with the same data, token budget, prompts, and evaluation. Only the random seed changes. The first run favored full fine-tuning; the second favors LoRA.

That instability is evidence about the experiment. Riverside may have measured a fortunate initialization or example order rather than a dependable strategy. A result that survives repeated seeds gives the recipe credit. A result that changes winner gives luck part of the credit and stays out of production.

### 3. A 2% Gain at Twice the Cost: Is It Worth Buying?

Full fine-tuning improves the required editing pass rate by 2%, but doubles training cost, artifact size, and deployment burden. The quality number is real; the business value is still undecided.

Effect size is the practical question: **is the improvement large enough to pay for?** If that 2% fixes a critical legal or safety failure, the answer may be yes. If it changes two harmless wording preferences, LoRA may remain the better engineering decision. Riverside compares quality and burden together instead of treating any positive delta as a win.

### 4. Two Editors Disagree: Clarify “Better” Before Agreement Math

One editor rewards vivid prose. Another rewards strict source support. Both are competent, but they are answering different questions.

Before calculating agreement, Riverside rewrites the rubric with concrete examples: contract compliance first, source support second, editorial usefulness third. Then the editors judge again. Agreement statistics can reveal whether a clear task is being applied consistently; they cannot repair an ambiguous definition of “better.”

These stories lead into the guidebook below: unstable evidence becomes `REVIEW`, repeatable evidence earns more trust, practical value must justify cost, and human judgments need a shared rubric before they can gate a release.

## Decision Guidebook: From Evidence to Release

### 1. State One Falsifiable Claim

For Riverside's editing assistant:

> On a versioned held-out editing suite, the immutable SFT-LoRA candidate will satisfy the instruction, source-support, and safety requirements while staying within Riverside's product-owned latency and cost limits.

This claim does not require prose perplexity or DPO preference. A later DPO claim would add: *blinded editors prefer DPO to the accepted SFT model while every SFT gate still passes*.

### 2. Let Each Failure Define One Observation

| Observable failure | Measurement | Gate interpretation |
| --- | --- | --- |
| Wrong format or stopping behavior | Instruction pass rate | Required floor |
| Invented or contradicted manuscript fact | Source-support rate | Critical unsupported claims block release |
| Prohibited behavior | Versioned safety pass rate | Critical failure blocks release |
| Slow requests interrupt editors | p95 latency | Required ceiling on target hardware and concurrency |
| Serving is unaffordable | Cost per completed request | Required operating ceiling |
| Two valid answers differ in usefulness | Blinded preference win rate | Enable only for a DPO improvement claim |

Product owners set thresholds before outputs are inspected. The notebook's numeric defaults illustrate policy shape; they are not Riverside requirements.

### 3. Write the Conclusion the Evidence Earned

- **Supported:** every required gate passes on the declared suite and candidate version.
- **Not supported:** any required gate fails.
- **Inconclusive:** required evidence is missing, contaminated, or too uncertain for the decision.

For the current teaching artifacts, the conclusion is **inconclusive for production**. SFT-LoRA advances to a real editing benchmark because its objective matches the workload. DPO remains an experiment; continuation candidates need a matched retraining study.

### 4. Turn the Conclusion into an Action

| Conclusion | Action |
| --- | --- |
| Offline gates fail | Keep the accepted release and diagnose the failed cases |
| Offline gates pass | Record artifact, suite, code, policy, and metrics; begin a small canary |
| Live gates degrade | Route back to the accepted immutable artifact |
| Canary remains healthy | Increase traffic under the same monitored gates |

A canary exists because offline cases cannot represent every live prompt length, phrasing, or traffic pattern. The deeper [LLM evaluation arc](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) covers suite construction, difficult slices, evaluator calibration, agreement, uncertainty, and adversarial testing.

### Apply the Guidebook in Code

The disabled workflow below loads external benchmark results, applies only the gates enabled for this SFT release claim, and writes a reproducible decision record. Teaching probes from this notebook are never copied into the release decision.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
from typing import Any, Mapping, Optional


RUN_PRODUCTION_DECISION = False


@dataclass(frozen=True)
class EvaluationPolicy:
    """Illustrative policy shape; replace every enabled default with a product-owned requirement."""

    min_instruction_pass_rate: Optional[float] = 0.95
    min_source_support_rate: Optional[float] = 0.98
    min_preference_win_rate: Optional[float] = None
    min_safety_pass_rate: Optional[float] = 1.0
    max_p95_latency_ms: Optional[float] = 1_500.0
    max_cost_per_1k_requests_usd: Optional[float] = 1.00
    max_perplexity_regression_pct: Optional[float] = None


@dataclass(frozen=True)
class ProductionDecisionConfig:
    workload: str = "editing-assistant"
    candidate_name: str = "Instruction-tuned (LoRA)"
    candidate_artifact: Path = CHECKPOINT_DIR / "instruction-lora"
    rollback_name: str = "previous-production"
    rollback_artifact: Path = Path("./artifacts/production/current")
    benchmark_metrics: Path = Path("./artifacts/production-benchmarks.json")
    registry_dir: Path = Path("./artifacts/finetuning-decisions")
    policy: EvaluationPolicy = EvaluationPolicy()


def evaluate_release(
    candidate: Mapping[str, float],
    baseline: Mapping[str, float],
    policy: EvaluationPolicy,
) -> dict[str, bool]:
    """Apply only gates enabled for the declared workload claim."""
    gates: dict[str, bool] = {}

    floor_gates = {
        "instruction": ("instruction_pass_rate", policy.min_instruction_pass_rate),
        "source_support": ("source_support_rate", policy.min_source_support_rate),
        "preference": ("preference_win_rate", policy.min_preference_win_rate),
        "safety": ("safety_pass_rate", policy.min_safety_pass_rate),
    }
    for gate_name, (metric_name, threshold) in floor_gates.items():
        if threshold is not None:
            gates[gate_name] = candidate[metric_name] >= threshold

    ceiling_gates = {
        "latency": ("p95_latency_ms", policy.max_p95_latency_ms),
        "cost": ("cost_per_1k_requests_usd", policy.max_cost_per_1k_requests_usd),
    }
    for gate_name, (metric_name, threshold) in ceiling_gates.items():
        if threshold is not None:
            gates[gate_name] = candidate[metric_name] <= threshold

    if policy.max_perplexity_regression_pct is not None:
        allowed_perplexity = baseline["heldout_perplexity"] * (
            1.0 + policy.max_perplexity_regression_pct / 100.0
        )
        gates["perplexity"] = candidate["heldout_perplexity"] <= allowed_perplexity

    if not gates:
        raise ValueError("At least one release gate must be enabled.")
    return gates


def sha256_artifact(path: Path) -> str:
    """Digest one checkpoint file or directory deterministically."""
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for file_path in files:
        relative_path = file_path.name if path.is_file() else file_path.relative_to(path).as_posix()
        digest.update(relative_path.encode("utf-8"))
        with file_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def build_decision_record(
    config: ProductionDecisionConfig,
    benchmark: Mapping[str, Any],
    gates: Mapping[str, bool],
    artifact_digest: str,
) -> dict[str, Any]:
    """Capture the claim, evidence, decision, and rollback target."""
    promoted = all(gates.values())
    return {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "workload": config.workload,
        "hypothesis": benchmark["hypothesis"],
        "conclusion": "supported" if promoted else "not_supported",
        "decision": "canary" if promoted else "keep_accepted_release",
        "candidate": {
            "name": config.candidate_name,
            "artifact": str(config.candidate_artifact),
            "sha256": artifact_digest,
        },
        "rollback": {
            "name": config.rollback_name,
            "artifact": str(config.rollback_artifact),
        },
        "lineage": {
            "dataset_fingerprint": benchmark["dataset_fingerprint"],
            "code_revision": benchmark["code_revision"],
            "base_model": MODEL_NAME,
        },
        "policy": asdict(config.policy),
        "candidate_metrics": benchmark["candidate"],
        "baseline_metrics": benchmark["baseline"],
        "gates": dict(gates),
    }


def write_decision_record(record: Mapping[str, Any], registry_dir: Path) -> Path:
    """Write a content-addressed decision record."""
    registry_dir.mkdir(parents=True, exist_ok=True)
    canonical = json.dumps(record, sort_keys=True, separators=(",", ":"))
    decision_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    output_path = registry_dir / f"decision-{decision_id}.json"
    output_path.write_text(json.dumps(record, indent=2, sort_keys=True), encoding="utf-8")
    return output_path

In [ ]:
production_config = ProductionDecisionConfig()

if RUN_PRODUCTION_DECISION:
    benchmark = json.loads(
        production_config.benchmark_metrics.read_text(encoding="utf-8")
    )
    if not production_config.candidate_artifact.exists():
        raise FileNotFoundError(
            f"Candidate artifact not found: {production_config.candidate_artifact}"
        )

    gates = evaluate_release(
        benchmark["candidate"],
        benchmark["baseline"],
        production_config.policy,
    )
    record = build_decision_record(
        production_config,
        benchmark,
        gates,
        sha256_artifact(production_config.candidate_artifact),
    )
    record_path = write_decision_record(record, production_config.registry_dir)

    print("Hypothesis:", record["hypothesis"])
    for gate_name, passed in gates.items():
        print(f"{gate_name:>15}: {'PASS' if passed else 'FAIL'}")
    print("Conclusion:", record["conclusion"])
    print("Decision:", record["decision"])
    print("Decision record:", record_path)
else:
    print(
        "Production decision workflow is disabled. Supply a versioned benchmark, "
        "product-owned thresholds, candidate artifact, and rollback artifact before enabling it."
    )